This is a starter notebook for the project, you'll have to import the libraries you'll need, you can find a list of the ones available in this workspace in the requirements.txt file in this workspace. 

In [14]:
!pip3 install -r requirements.txt

  Using cached langchain-0.0.305-py3-none-any.whl (1.8 MB)
  Using cached pytest-8.3.2-py3-none-any.whl (341 kB)
  Using cached sentence_transformers-3.0.1-py3-none-any.whl (227 kB)
  Using cached transformers-4.44.2-py3-none-any.whl (9.5 MB)
  Using cached jupyter-1.0.0-py2.py3-none-any.whl (2.7 kB)
     |████████████████████████████████| 80 kB 1.1 MB/s eta 0:00:01
     |████████████████████████████████| 2.1 MB 4.1 MB/s eta 0:00:01
     |████████████████████████████████| 56 kB 15.3 MB/s eta 0:00:01
  Using cached numexpr-2.10.1-cp39-cp39-macosx_11_0_arm64.whl (130 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl (12 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl (28 kB)
  Using cached async_timeout-4.0.3-py3-none-any.whl (5.7 kB)
  Using cached aiohttp-3.10.5-cp39-cp39-macosx_11_0_arm64.whl (389 kB)
  Using cached ipykernel-6.29.5-py3-none-any.whl (117 kB)
     |████████████████████████████████| 123 kB 76.6 MB/s eta 0:00:01
  Using cached jupyter_console-6.6.3-py3-no

In [29]:
from langchain_community.chat_models import ChatOpenAI 
from langchain.prompts import FewShotPromptTemplate, PromptTemplate
from pydantic import BaseModel, Field, NonNegativeInt
from comet_ml import Experiment
from fastapi.encoders import jsonable_encoder
from diffusers import StableDiffusionPipeline
# import openai
import time
from tqdm import tqdm
import google.generativeai as genai
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings
import google.generativeai as genai
import os
from dotenv import load_dotenv
import pandas as pd
from typing import List
import requests

# Load environment variables
load_dotenv('my_config.env')

# API configuration
API_KEY = os.getenv('API_KEY')
# openai.api_key = os.getenv("OPENAI_API_KEY")
COMET_API_KEY = os.getenv("COMET_API_KEY")


# Initialize the experiment
experiment = Experiment(
    api_key=COMET_API_KEY,
    project_name="real-estate-agent",
    workspace="polarbeargo",
    log_code=True,
)

genai.configure(api_key=os.getenv('API_KEY'))

safety_settings = [
    {
        "category": "HARM_CATEGORY_DANGEROUS_CONTENT",
        "threshold": "BLOCK_NONE",
    },
]

model = genai.GenerativeModel('models/gemini-1.5-flash', safety_settings=safety_settings)


COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/polarbeargo/real-estate-agent/e2dbd3386402480b8083487d7d16767b



- Define the prompt for generating synthetic real estate data (images and text)

In [30]:
model_name = "gpt-3.5-turbo"
llm = ChatOpenAI(model_name=model_name, temperature=0.7)

# Initialize the diffusion model for image generation
image_model = StableDiffusionPipeline.from_pretrained("CompVis/stable-diffusion-v1-4")  # Replace with the appropriate model

class Listing(BaseModel):
    neighborhood: str = Field()
    price: NonNegativeInt = Field()
    bedrooms: NonNegativeInt = Field()
    bathrooms: NonNegativeInt = Field()
    house_size: NonNegativeInt = Field()
    description: str = Field()
    neighborhood_description: str = Field()

def create_listing_prompt(listing: Listing) -> str:
    return f"""
    Neighborhood: {listing.neighborhood}
    Price: ${listing.price}
    Bedrooms: {listing.bedrooms}
    Bathrooms: {listing.bathrooms}
    House Size: {listing.house_size} sqft

    Description: {listing.description}

    Neighborhood Description: {listing.neighborhood_description}
    """

examples = [
    {
        "question": "Generate a listing for a 3-bedroom house in downtown.",
        "answer": Listing(
            neighborhood="Downtown",
            price=500000,
            bedrooms=3,
            bathrooms=2,
            house_size=1500,
            description="A beautiful 3-bedroom house located in the heart of downtown with modern amenities.",
            neighborhood_description="Downtown is vibrant and bustling, with plenty of restaurants, shops, and parks."
        )
    },
    {
        "question": "Create a listing for a luxury apartment in the suburbs.",
        "answer": Listing(
            neighborhood="Suburbia",
            price=750000,
            bedrooms=2,
            bathrooms=2,
            house_size=1200,
            description="A luxurious apartment featuring high-end finishes and spacious living areas.",
            neighborhood_description="Suburbia offers a peaceful environment with great schools and family-friendly parks."
        )
    }
]

example_prompt = PromptTemplate(
    input_variables=["question", "answer"],
    template="{question}\n{answer}"
)

few_shot_prompt = FewShotPromptTemplate(
    examples=[{"question": ex["question"], "answer": create_listing_prompt(ex["answer"])} for ex in examples],
    example_prompt=example_prompt,
    suffix="Use these examples to generate a listing for the following question: {input}",
    input_variables=["input"]
)

def generate_image(prompt):
    image = image_model(prompt).images[0]
    return image

def chain_of_thoughts(questions: List[str]) -> List[str]:
    responses = []
    
    for question in questions:
        full_prompt = few_shot_prompt.format(input=question)
        print(f"Full Prompt: {full_prompt} (Type: {type(full_prompt)})")
        
        response = llm(full_prompt)
        responses.append(response)
        
        image_prompt = f"Generate an image for a {response.bedrooms}-bedroom house in {response.neighborhood}."
        image = generate_image(image_prompt)
        image.save(f"{response.neighborhood}_listing.png")

    return responses

questions = [
    "Generate a listing for a 4-bedroom house near the beach.",
    "Create a listing for a cozy studio apartment in the city."
]

listings = chain_of_thoughts(questions)
for listing in listings:
    print(listing)

listings_df = pd.DataFrame([jsonable_encoder(listing) for listing in listings])
listings_df.head()

Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

(…)kpoints/scheduler_config-checkpoint.json:   0%|          | 0.00/209 [00:00<?, ?B/s]

text_encoder/config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

/Users/hsin-wenchang/Documents/GitHub/GenAIND-Project-Personalized-Real-Estate-Agent/env/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Full Prompt: Generate a listing for a 3-bedroom house in downtown.

    Neighborhood: Downtown
    Price: $500000
    Bedrooms: 3
    Bathrooms: 2
    House Size: 1500 sqft

    Description: A beautiful 3-bedroom house located in the heart of downtown with modern amenities.

    Neighborhood Description: Downtown is vibrant and bustling, with plenty of restaurants, shops, and parks.
    

Create a listing for a luxury apartment in the suburbs.

    Neighborhood: Suburbia
    Price: $750000
    Bedrooms: 2
    Bathrooms: 2
    House Size: 1200 sqft

    Description: A luxurious apartment featuring high-end finishes and spacious living areas.

    Neighborhood Description: Suburbia offers a peaceful environment with great schools and family-friendly parks.
    

Use these examples to generate a listing for the following question: Generate a listing for a 4-bedroom house near the beach. (Type: <class 'str'>)


TypeError: Got unknown type G

In [ ]:
listings_df.to_csv('listings.csv', index=False)

In [ ]:
questions = [   
                "How big do you want your house to be?" 
                "What are 3 most important things for you in choosing this property?", 
                "Which amenities would you like?", 
                "Which transportation options are important to you?",
                "How urban do you want your neighborhood to be?",   
            ]
answers = [
    "A comfortable three-bedroom house with a spacious kitchen and a cozy living room.",
    "A quiet neighborhood, good local schools, and convenient shopping options.",
    "A backyard for gardening, a two-car garage, and a modern, energy-efficient heating system.",
    "Easy access to a reliable bus line, proximity to a major highway, and bike-friendly roads.",
    "A balance between suburban tranquility and access to urban amenities like restaurants and theaters."
]

### Multimodal Vector Store, Embeddings and Search

In [ ]:
class GeminiEmbeddingFunction(EmbeddingFunction):
  def __call__(self, input: Documents) -> Embeddings:
    model = 'models/text-embedding-004'
    title = "Custom query"
    return genai.embed_content(model=model,
                                content=input,
                                task_type="retrieval_document",
                                title=title)["embedding"]

In [31]:
def create_chroma_db(documents, name):
  chroma_client = chromadb.Client()
  db = chroma_client.get_or_create_collection(name=name, embedding_function=GeminiEmbeddingFunction())

  for i, d in enumerate(documents):
    db.add(
      documents=d,
      ids=str(i)
    )
  return db

- Storing Listings Into a Vector Database

In [26]:
from langchain.document_loaders.csv_loader import CSVLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

loader = CSVLoader(file_path='./real_estate_listings.csv')
docs = loader.load()

splitter =  RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=100,
    length_function=len,
    add_start_index=True,
)
split_docs = splitter.split_documents(docs)

# Save split documents to ChromaDB
db = create_chroma_db(split_docs, "split_documents")

NameError: name 'CSVLoader' is not defined

- Integrate kubeflow pipelines

In [ ]:
%%writefile HomeMatch.py


- Use Gradio to create an interactive interface where we can input data related to home preferences, and the model can predict the best match for them.

In [ ]:
import gradio as gr


In [ ]:

def generate_listing(neighborhood, price, bedrooms, bathrooms, house_size, description, neighborhood_description):
        return {
            "Neighborhood": neighborhood,
            "Price": price,
            "Bedrooms": bedrooms,
            "Bathrooms": bathrooms,
            "House Size": house_size,
            "Description": description,
            "Neighborhood Description": neighborhood_description
        }

def predict_best_match(neighborhood, price, bedrooms, bathrooms, house_size, description, neighborhood_description):
    listing = generate_listing(neighborhood, price, bedrooms, bathrooms, house_size, description, neighborhood_description)
    # Here you can add the logic to predict the best match using the model
    return listing

interface = gr.Interface(
    fn=predict_best_match,
    inputs=[
        gr.inputs.Textbox(label="Neighborhood"),
        gr.inputs.Number(label="Price"),
        gr.inputs.Number(label="Bedrooms"),
        gr.inputs.Number(label="Bathrooms"),
        gr.inputs.Number(label="House Size"),
        gr.inputs.Textbox(label="Description"),
        gr.inputs.Textbox(label="Neighborhood Description")
    ],
    outputs="json",
    title="Home Match Predictor",
    description="Input your home preferences to find the best match."
)

In [ ]:
if __name__ == "__main__":
    interface.launch()

In [32]:
experiment.end()

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : far_raccoon_441
COMET INFO:     url                   : https://www.comet.com/polarbeargo/real-estate-agent/e2dbd3386402480b8083487d7d16767b
COMET INFO:   Uploads:
COMET INFO:     environment details      : 1
COMET INFO:     filename                 : 1
COMET INFO:     git metadata             : 1
COMET INFO:     git-patch (uncompressed) : 1 (66.14 KB)
COMET INFO:     installed packages       : 1
COMET INFO:     notebook                 : 1
COMET INFO:     source_code              : 1
COMET INFO: 
COMET INFO: Please wait for assets to finish uploading (timeout is 10800 seconds)
COMET INFO: All assets have been sent, waiting for delivery confirmation
